# 04 — Auto Min/Max Reorder Point Logic

The last link in the chain: turn the inventory simulation output (notebook 03) into a concrete, per-SKU reorder recommendation — reorder point (min), max level, and suggested order quantity (via EOQ). See `src/reorder_logic.py`.

The "auto" part: this whole table is designed to be **recalculated on every forecast refresh** (e.g. nightly, once new sales data lands and Prophet reforecasts) rather than being a one-time manually-set min/max — that's the actual operational upgrade over a static min/max policy.

**Caveat, stated plainly:** the public dataset has no real inventory ledger, so on-hand inventory here is synthetic (`simulate_synthetic_on_hand`, clearly labeled). Every other piece — reorder point, EOQ, min/max, the trigger logic — is real and would plug into an actual inventory feed unchanged.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from src.reorder_logic import build_reorder_table, simulate_synthetic_on_hand

sim_results = pd.read_parquet(Path.cwd().parent / "data" / "processed" / "inventory_simulation.parquet")
sim_results.head()

In [ ]:
series_keys = list(sim_results[["store", "item"]].itertuples(index=False, name=None))
expected_demand_lookup = {
    (row.store, row.item): row.expected_lead_time_demand / row.lead_time_days
    for row in sim_results.itertuples()
}

on_hand = simulate_synthetic_on_hand(series_keys, expected_demand_lookup)
on_hand.head()

## Build the reorder table

`order_cost` and `holding_cost_per_unit` are placeholder procurement assumptions (documented, swappable) used only for the EOQ order-quantity calculation.

In [ ]:
reorder_table = build_reorder_table(
    sim_results, on_hand, order_cost=50.0, holding_cost_per_unit=2.0
)
print(reorder_table.shape)
print(f"{reorder_table['needs_reorder'].sum()} of {len(reorder_table)} SKUs currently need reordering")
reorder_table.sort_values("needs_reorder", ascending=False).head(15)

## Visualize on-hand vs. min/max for a sample of SKUs

The chart that makes the policy legible at a glance: on-hand bar against its min (reorder point) and max markers. This is the visual for the Power BI "Reorder Queue" page.

In [ ]:
sample = reorder_table.sample(15, random_state=1).sort_values("on_hand")
labels = [f"s{s}-i{i}" for s, i in zip(sample["store"], sample["item"])]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(labels, sample["on_hand"], color="#4C72B0", label="on hand")
ax.scatter(sample["min_level"], labels, color="orange", marker="|", s=200, label="min (reorder point)")
ax.scatter(sample["max_level"], labels, color="green", marker="|", s=200, label="max level")
ax.legend(); ax.set_xlabel("units")
ax.set_title("On-hand vs. min/max levels — sample of SKUs")
plt.tight_layout()

## Export final tables for Power BI

This is the table the "Reorder Queue" dashboard page filters/sorts directly. See `powerbi/README.md` for the full data model this joins into.

In [ ]:
reorder_table.to_csv(Path.cwd().parent / "outputs" / "reorder_recommendations.csv", index=False)